Auditoría de Datos, EDA y Deduplicación

Este cuaderno implementa la Fase 1 del plan de choque. Antes de aplicar ninguna limpieza o hash difuso, realizamos un Análisis Exploratorio de Datos (EDA) estricto sobre los tres archivos `raw` originales para:
1. Validar la integridad de los esquemas (columnas) entre versiones.
2. Identificar variables de segmentación ignoradas en el pipeline anterior (ej. `business_type`).
3. Evaluar el solapamiento de registros entre los ficheros antes de la concatenación.

In [4]:
import pandas as pd
from pathlib import Path

# Definición de rutas relativas asumiendo ejecución desde la carpeta /notebook
RUTA_RAW = Path("../data/raw")

archivos = {
    'v1_50': 'aa_dataset-tickets-multi-lang-5-2-50-version.csv',
    'v2_20k': 'dataset-tickets-multi-lang-4-20k.csv',
    'v3_4k': 'dataset-tickets-multi-lang3-4k.csv'
}

# Carga en memoria y auditoría de esquemas
dfs = {}
for nombre, archivo in archivos.items():
    ruta = RUTA_RAW / archivo
    if ruta.exists():
        # Usamos low_memory=False por si hay tipos mixtos en las columnas
        dfs[nombre] = pd.read_csv(ruta, low_memory=False)
        print(f"--- Dataset: {nombre} ---")
        print(f"Filas: {dfs[nombre].shape[0]} | Columnas: {dfs[nombre].shape[1]}")
        print(f"Esquema: {list(dfs[nombre].columns)}\n")
    else:
        print(f"Error: No se pudo localizar {ruta}")

--- Dataset: v1_50 ---
Filas: 28587 | Columnas: 16
Esquema: ['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language', 'version', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8']

--- Dataset: v2_20k ---
Filas: 20000 | Columnas: 15
Esquema: ['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8']

--- Dataset: v3_4k ---
Filas: 4000 | Columnas: 17
Esquema: ['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language', 'business_type', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8', 'tag_9']



Auditoría de variables asimétricas y solapamiento

Inspeccionamos la distribución de `business_type` en el único fichero que la contiene para descartarla formalmente del pipeline por falta de soporte. 

A continuación, cuantificamos el solapamiento exacto de textos entre los tres ficheros originales. Esto medirá la cantidad de duplicados puros inyectados en el corpus antes siquiera de aplicar el Hash Difuso para los casi-duplicados.

In [5]:
# Justificación para descartar business_type
print("--- Distribución de 'business_type' en v3_4k ---")
print(dfs['v3_4k']['business_type'].value_counts(dropna=False))

# Análisis de solapamiento exacto (fuga de datos estructural)
# Comparamos el contenido exacto de la columna 'body' ignorando nulos
cuerpos_v1 = set(dfs['v1_50']['body'].dropna())
cuerpos_v2 = set(dfs['v2_20k']['body'].dropna())
cuerpos_v3 = set(dfs['v3_4k']['body'].dropna())

print("\n--- Textos únicos por fichero ---")
print(f"Textos en v1_50: {len(cuerpos_v1)}")
print(f"Textos en v2_20k: {len(cuerpos_v2)}")
print(f"Textos en v3_4k:  {len(cuerpos_v3)}")

# Intersecciones
solap_v1_v2 = len(cuerpos_v1.intersection(cuerpos_v2))
solap_v1_v3 = len(cuerpos_v1.intersection(cuerpos_v3))
solap_v2_v3 = len(cuerpos_v2.intersection(cuerpos_v3))

print("\n--- Solapamiento cruzado ---")
print(f"Textos de v2_20k que ya están en v1_50: {solap_v1_v2}")
print(f"Textos de v3_4k que ya están en v1_50:  {solap_v1_v3}")
print(f"Textos de v3_4k que ya están en v2_20k: {solap_v2_v3}")

--- Distribución de 'business_type' en v3_4k ---
business_type
IT Services                     1716
Tech Online Store               1395
IT Consulting Firm               391
Software Development Company     311
Online Store                     171
IT Consulting Service              7
Pit Services                       5
Adobe Photoshop 2024               2
_IT_Services_                      2
Name: count, dtype: int64

--- Textos únicos por fichero ---
Textos en v1_50: 28587
Textos en v2_20k: 19998
Textos en v3_4k:  3999

--- Solapamiento cruzado ---
Textos de v2_20k que ya están en v1_50: 8399
Textos de v3_4k que ya están en v1_50:  0
Textos de v3_4k que ya están en v2_20k: 0


Consolidación y Filtros de Dominio

Se unifican los ficheros proyectando únicamente las columnas estructurales compartidas. A continuación, se aplican los filtros de negocio para acotar el problema al dominio establecido: retención exclusiva de tickets en inglés y filtrado de las colas operativas del sector Telco. 

Por último, se aplica una deduplicación exacta sobre el texto completo (asunto y cuerpo) para eliminar el solapamiento inter-ficheros detectado y aligerar la carga computacional del posterior hash difuso.

In [6]:
columnas_relevantes = ['subject', 'body', 'queue', 'type', 'priority', 'language']

# Proyección y concatenación
df_concat = pd.concat([
    dfs['v1_50'][columnas_relevantes],
    dfs['v2_20k'][columnas_relevantes],
    dfs['v3_4k'][columnas_relevantes]
], ignore_index=True)

# Tratamiento de nulos estructurales
df_concat['subject'] = df_concat['subject'].fillna('No Subject')
df_concat = df_concat.dropna(subset=['body'])

# Filtros de negocio (Inglés y Sector Telco)
df_telco = df_concat[df_concat['language'] == 'en'].copy()

colas_telco = [
    'Technical Support', 'Service Outages and Maintenance', 
    'Billing and Payments', 'Sales and Pre-Sales', 'Customer Service'
]
df_telco = df_telco[df_telco['queue'].isin(colas_telco)]
df_telco = df_telco.drop(columns=['language'])

# Fusión para evaluación de similitud
df_telco['full_text'] = df_telco['subject'] + ". " + df_telco['body']

# Deduplicación exacta (limpieza del solapamiento evidente)
vol_pre = len(df_telco)
df_telco = df_telco.drop_duplicates(subset=['full_text'], keep='first')
vol_post = len(df_telco)

print(f"Volumen unificado tras filtros de negocio: {vol_pre} tickets")
print(f"Volumen retenido tras deduplicación exacta: {vol_post} tickets")
print(f"Duplicados exactos destruidos: {vol_pre - vol_post}")

Volumen unificado tras filtros de negocio: 18158 tickets
Volumen retenido tras deduplicación exacta: 15403 tickets
Duplicados exactos destruidos: 2755


Agrupación de Casi-Duplicados (Fuzzy Hashing)

Para evitar fugas de datos (Data Leakage) en las particiones de entrenamiento y test, se agrupan los tickets que comparten estructuras léxicas casi idénticas (plantillas). Se extraen n-gramas del cuerpo del ticket y se aplica el algoritmo MinHash con Locality-Sensitive Hashing (LSH) para detectar similitudes superiores al umbral establecido, asignando a cada ticket un identificador de clúster único.

In [7]:
import re
# pip install datasketch
from datasketch import MinHash, MinHashLSH

def limpiar_texto_hash(texto):
    """Normalización estricta para la extracción de características del hash."""
    texto = str(texto).lower()
    texto = re.sub(r'\s+', ' ', texto)
    texto = re.sub(r'[^\w\s]', '', texto)
    return texto.strip()

def obtener_shingles(texto, n_gram=3):
    """Tokenización por n-gramas solapados para capturar contexto estructural."""
    tokens = texto.split()
    if len(tokens) < n_gram:
        return set(tokens)
    return set([' '.join(tokens[i:i+n_gram]) for i in range(len(tokens) - n_gram + 1)])

UMBRAL_SIMILITUD = 0.85
NUM_PERM = 128

print(f"Generando firmas MinHash al {UMBRAL_SIMILITUD*100}% de similitud...")
lsh = MinHashLSH(threshold=UMBRAL_SIMILITUD, num_perm=NUM_PERM)
firmas = {}

# Reset de índice necesario para mapear correctamente con LSH
df_telco = df_telco.reset_index(drop=True)

# 1. Extracción e ingesta en LSH
for idx, row in df_telco.iterrows():
    texto_limpio = limpiar_texto_hash(row['body'])
    shingles = obtener_shingles(texto_limpio)
    
    m = MinHash(num_perm=NUM_PERM)
    for s in shingles:
        m.update(s.encode('utf8'))
        
    lsh.insert(str(idx), m)
    firmas[str(idx)] = m

# 2. Agrupación por vecindarios
print("Calculando intersecciones y asignando clústeres...")
visitados = set()
mapa_clusters = {}
id_cluster = 0

for idx in firmas.keys():
    if idx not in visitados:
        vecinos = lsh.query(firmas[idx])
        for v in vecinos:
            visitados.add(v)
            mapa_clusters[int(v)] = f"cluster_{id_cluster}"
        id_cluster += 1

df_telco['cluster_id'] = df_telco.index.map(mapa_clusters)

print(f"\n--- Resultados del Hash Difuso ---")
print(f"Tickets analizados: {len(df_telco)}")
print(f"Clústeres únicos (plantillas reales): {id_cluster}")
print(f"Reducción lograda: {len(df_telco) - id_cluster} tickets eran casi-duplicados")

Generando firmas MinHash al 85.0% de similitud...
Calculando intersecciones y asignando clústeres...

--- Resultados del Hash Difuso ---
Tickets analizados: 15403
Clústeres únicos (plantillas reales): 15241
Reducción lograda: 162 tickets eran casi-duplicados


Construcción de la Variable Objetivo y Clases Minoritarias

Se concatena la jerarquía de tipificación para formar la clase objetivo completa (Cola + Tipo + Prioridad). Para garantizar la representatividad estadística en las particiones de los modelos de validación cruzada, se audita el soporte de cada clase contando el número de clústeres independientes, no los tickets individuales.

Las tripletas con un soporte inferior al umbral mínimo de validación se colapsan a la categoría genérica `OUT_OF_SCOPE`. Estas observaciones se evaluarán siempre como "no automatizables" en el simulador financiero.

In [8]:
# Fusión de la clase objetivo
df_telco['target_tripleta'] = (
    df_telco['queue'].astype(str) + '_' + 
    df_telco['type'].astype(str) + '_' + 
    df_telco['priority'].astype(str)
)

# Auditoría de soporte basado estrictamente en la varianza estructural (clústeres)
soporte_por_clase = df_telco.groupby('target_tripleta')['cluster_id'].nunique()

MIN_SAMPLES = 20

clases_validas = soporte_por_clase[soporte_por_clase >= MIN_SAMPLES].index
clases_descartadas = soporte_por_clase[soporte_por_clase < MIN_SAMPLES].index

# Aplicación del colapso de clases raras
OUT_OF_SCOPE_LABEL = 'OUT_OF_SCOPE'
df_telco['target_tripleta'] = df_telco['target_tripleta'].apply(
    lambda x: x if x in clases_validas else OUT_OF_SCOPE_LABEL
)

# Resumen analítico
print(f"Total de combinaciones únicas observadas: {len(soporte_por_clase)}")
print(f"Clases retenidas (soporte >= {MIN_SAMPLES} clústeres): {len(clases_validas)}")
print(f"Clases descartadas por bajo soporte: {len(clases_descartadas)}")

out_of_scope_tickets = len(df_telco[df_telco['target_tripleta'] == OUT_OF_SCOPE_LABEL])
out_of_scope_pct = (out_of_scope_tickets / len(df_telco)) * 100

print(f"\nTickets desplazados a {OUT_OF_SCOPE_LABEL}: {out_of_scope_tickets} ({out_of_scope_pct:.2f}%)")

print("\nTop 5 Clases Mayoritarias:")
print(df_telco['target_tripleta'].value_counts().head())

Total de combinaciones únicas observadas: 60
Clases retenidas (soporte >= 20 clústeres): 55
Clases descartadas por bajo soporte: 5

Tickets desplazados a OUT_OF_SCOPE: 71 (0.46%)

Top 5 Clases Mayoritarias:
target_tripleta
Technical Support_Incident_high      2262
Technical Support_Incident_medium    1115
Technical Support_Problem_high        884
Technical Support_Request_high        857
Customer Service_Request_medium       828
Name: count, dtype: int64


Exportación de la Capa Gold (Corpus de Entrenamiento)

Con la variable objetivo consolidada, el ruido estructural eliminado y el mapeo de clústeres para prevenir fugas de datos finalizado, se congela el estado del dataframe. 

Se proyectan únicamente las variables estrictamente necesarias para la tubería de Machine Learning (el texto unificado, la tripleta objetivo y el identificador de clúster) y se persiste en formato Parquet para garantizar compresión y tipado estricto en la lectura de los siguientes cuadernos.

In [9]:
import os

# Creación de directorio si fue purgado
GOLD_DIR = "../data/gold"
os.makedirs(GOLD_DIR, exist_ok=True)

# Proyección de variables para entrenamiento
columnas_modelo = ['full_text', 'target_tripleta', 'cluster_id']
df_gold = df_telco[columnas_modelo].copy()

# Persistencia estática
ruta_export = os.path.join(GOLD_DIR, 'corpus_sitor_limpio.parquet')

# Usamos fastparquet (si da error, instala pip install fastparquet)
df_gold.to_parquet(ruta_export, engine='fastparquet', index=False)

print("--- Pipeline de Preparación Finalizado ---")
print(f"Ruta de exportación: {ruta_export}")
print(f"Total de registros listos para particionar: {len(df_gold)}")

--- Pipeline de Preparación Finalizado ---
Ruta de exportación: ../data/gold\corpus_sitor_limpio.parquet
Total de registros listos para particionar: 15403
